# ArbGraph × RAG-Conflicts

**Pipeline:** `rag_conflicts` dataset (passages pre-retrieved) → ArbGraph (Atomization → Alignment → Evidence Graph → Arbitration → Generation)

**Option A:** Dùng `search_results` có sẵn trong dataset thay vì Wikipedia retriever.

```
conflicts.jsonl schema:
  question        : str
  search_results  : [{title, url, response_str, short_text, snippet, date}]
  conflict_type   : str   (inter_source | extra_source | parametric | no_conflict)
  correct_answer  : str
```

## Cell 1 — Install dependencies

In [ ]:
import subprocess, sys

pkgs = [
    'torch>=2.0',
    'transformers>=4.51.0',
    'sentence-transformers>=2.2',
    'networkx>=3.0',
    'numpy>=1.23',
    'tqdm>=4.60',
    'requests>=2.28',
    'accelerate>=0.26',
    'bitsandbytes>=0.41',
    'scikit-learn>=1.2',
    'pandas>=1.5',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('Done')

## Cell 2 — Clone ArbGraph + download dataset

In [ ]:
import os, sys

# ── ArbGraph repo ────────────────────────────────────────────────────────
REPO_DIR = '/content/ArbGraph'
if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/1212Judy/ArbGraph {REPO_DIR}')
    print('Cloned ArbGraph')
else:
    os.system(f'cd {REPO_DIR} && git pull')
    print('ArbGraph up to date')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── RAG-Conflicts dataset ─────────────────────────────────────────────────
DATA_DIR  = '/content/data/rag_conflicts'
DATA_FILE = f'{DATA_DIR}/conflicts.jsonl'
os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(DATA_FILE):
    url = 'https://raw.githubusercontent.com/google-research-datasets/rag_conflicts/main/conflicts.jsonl'
    os.system(f'wget -q "{url}" -O "{DATA_FILE}"')
    print(f'Downloaded conflicts.jsonl')
else:
    print(f'Dataset already exists')

# Verify
import json
with open(DATA_FILE) as f:
    first = json.loads(f.readline())
print(f'Keys: {list(first.keys())}')
print(f'Question: {first["question"]}')
print(f'Num search_results: {len(first["search_results"])}')
print(f'conflict_type: {first["conflict_type"]}')
print(f'correct_answer: {first.get("correct_answer", "N/A")}')

## Cell 3 — GPU check

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM    : {vram:.1f} GB')


## Cell 4 — HFAdapter (original + 4-bit)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

class HFAdapter:
    """
    Exact copy of local_llm/hf_adapter.py.
    Added: load_in_4bit for T4/P100 GPUs.
    """
    def __init__(self, model_name_or_path: str, load_in_4bit: bool = False):
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name_or_path, use_fast=True)

        if load_in_4bit and torch.cuda.is_available():
            bnb = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type='nf4',
                bnb_4bit_compute_dtype=torch.bfloat16,
            )
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name_or_path, device_map='auto', quantization_config=bnb).eval()
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name_or_path,
                device_map='auto',
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
            ).eval()

    def generate(self, prompt: str, max_new_tokens: int = 400, temperature: float = 0.0) -> str:
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)
        kwargs = {
            'max_new_tokens': max_new_tokens,
            'use_cache': True,
            'pad_token_id': self.tokenizer.eos_token_id,
            'eos_token_id': self.tokenizer.eos_token_id,
        }
        if temperature and temperature > 0:
            kwargs['do_sample'] = True
            kwargs['temperature'] = temperature
        else:
            kwargs['do_sample'] = False
        with torch.no_grad():
            outputs = self.model.generate(**inputs, **kwargs)
        tokens = outputs[0][inputs['input_ids'].shape[-1]:]
        return self.tokenizer.decode(tokens, skip_special_tokens=True)

print('HFAdapter ready')

## Cell 5 — Config
Chỉnh `MODEL_NAME` nếu muốn thử model khác.

In [ ]:
import os, torch

# ── Model ────────────────────────────────────────────────────────────────
# Qwen2.5-1.5B-Instruct: ~3x nhanh hơn 4B trên T4, vẫn đủ chất lượng cho pipeline
# Đổi lại 'Qwen/Qwen3-4B-Instruct-2507' nếu muốn chạy full experiment
MODEL_NAME = os.environ.get('MODEL_NAME', 'Qwen/Qwen2.5-1.5B-Instruct')

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
LOAD_IN_4BIT = vram_gb < 20

# ── Paths ─────────────────────────────────────────────────────────────────
DATA_FILE   = '/content/data/rag_conflicts/conflicts.jsonl'
OUTPUT_FILE = '/content/outputs/rag_conflicts/arbgraph_predictions.jsonl'
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# ── Hyperparams — paper defaults ──────────────────────────────────────────
MAX_ROUNDS          = 2
ACCEPT_THRESHOLD    = 0.3
ARBITRATION_BUDGET  = 3
STEP_SIZE           = 0.8
ENCODER_NAME        = 'BAAI/bge-large-en-v1.5'
SIMILARITY_THRESHOLD = 0.88
QUERY_RELEVANCE_THRESHOLD = 0.3
PAIR_SIMILARITY_THRESHOLD = 0.75
MAX_SUPPORT_EDGES         = 60

# ── Speed knobs ────────────────────────────────────────────────────────────
PASSAGE_FIELD = 'short_text'  # 'short_text' (512 tok) hoặc 'response_str' (full)
MAX_DOCS      = 5             # số passages tối đa mỗi câu
MAX_QUESTIONS = 10            # None = tất cả 458 câu

print(f'Model        : {MODEL_NAME}')
print(f'4-bit        : {LOAD_IN_4BIT}  (VRAM={vram_gb:.1f}GB)')
print(f'Passage field: {PASSAGE_FIELD}')
print(f'Max docs/q   : {MAX_DOCS}')
print(f'Max questions: {MAX_QUESTIONS}')
print(f'Output       : {OUTPUT_FILE}')


## Cell 6 — PreloadedRetriever

Thay thế `WikipediaRetriever` bằng class đọc `search_results` có sẵn trong mỗi record.  
Interface giữ nguyên: trả về list `Document` — các module gốc không thay đổi gì.

In [ ]:
from dataclasses import dataclass
from typing import List

@dataclass
class Document:
    """Same interface as retrieval/retriever.py:Document"""
    id: str
    text: str
    title: str
    url: str


class PreloadedRetriever:
    """
    Replaces WikipediaRetriever for Option A.
    Converts search_results from the dataset record into Document objects.
    The .retrieve() signature is identical to WikipediaRetriever.retrieve().
    """
    def __init__(self, search_results: list, passage_field: str = 'short_text'):
        """
        search_results : list of dicts from conflicts.jsonl
        passage_field  : 'short_text' | 'response_str'
        """
        self._docs = []
        for idx, sr in enumerate(search_results):
            text = sr.get(passage_field) or sr.get('response_str') or sr.get('snippet') or ''
            self._docs.append(Document(
                id    = f'sr_{idx}',
                text  = str(text).strip(),
                title = sr.get('title', ''),
                url   = sr.get('url', ''),
            ))

    def retrieve(self, query: str, max_docs: int = 10) -> List[Document]:
        """Returns pre-loaded documents, ignores query (already retrieved)."""
        return [d for d in self._docs if d.text][:max_docs]


print('PreloadedRetriever ready')

## Cell 7 — Load dataset

In [ ]:
import json
from collections import defaultdict

all_records = []
with open(DATA_FILE, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        rec['id']       = rec.get('id', str(i))
        rec['Question'] = rec['question']
        all_records.append(rec)

# Stratified sample: lấy đều từ mỗi conflict_type
if MAX_QUESTIONS and MAX_QUESTIONS < len(all_records):
    from collections import defaultdict
    buckets = defaultdict(list)
    for r in all_records:
        buckets[r['conflict_type']].append(r)

    conflict_types = sorted(buckets.keys())
    per_type = MAX_QUESTIONS // len(conflict_types)
    remainder = MAX_QUESTIONS % len(conflict_types)

    records = []
    for i, ct in enumerate(conflict_types):
        n = per_type + (1 if i < remainder else 0)
        records.extend(buckets[ct][:n])

    print(f'Stratified sample: {len(records)} câu từ {len(conflict_types)} conflict types')
    from collections import Counter
    for k, v in Counter(r['conflict_type'] for r in records).most_common():
        print(f'  {k:<40} {v}')
else:
    records = all_records
    from collections import Counter
    print(f'Full dataset: {len(records)} records')
    for k, v in Counter(r['conflict_type'] for r in records).most_common():
        print(f'  {k:<40} {v}')


## Cell 8 — Load model + pipeline components

In [ ]:
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from atomization          import ClaimExtractor
from claim_alignment      import NodeAligner
from evidence_graph       import EvidenceGraphBuilder
from conflict_arbitration import ArbGraphArbitrator
from longform_generation  import AnswerGenerator

print('Loading model (may take 5–10 min on first run)...')
llm = HFAdapter(MODEL_NAME, load_in_4bit=LOAD_IN_4BIT)
print(f'Model loaded: {MODEL_NAME}')

extractor     = ClaimExtractor(llm)
aligner       = NodeAligner(llm_engine=llm, encoder_name=ENCODER_NAME,
                             similarity_threshold=SIMILARITY_THRESHOLD)
graph_builder = EvidenceGraphBuilder(llm, embedding_model=ENCODER_NAME)
arbitrator    = ArbGraphArbitrator(llm=llm, max_rounds=MAX_ROUNDS,
                                   accept_threshold=ACCEPT_THRESHOLD,
                                   arbitration_budget=ARBITRATION_BUDGET,
                                   step_size=STEP_SIZE)
generator     = AnswerGenerator(llm)
print('All components ready')

## Cell 9 — Core pipeline functions
Exact copy từ `run_arbgraph.py`, chỉ thêm `retriever` như tham số (thay vì lấy từ components).

In [ ]:
import json, os
from typing import Dict


def extract_claims_from_documents(extractor, documents):
    """Exact copy from run_arbgraph.py"""
    claims = []
    for doc in documents:
        text = doc.text
        chunk_size = 2500
        step = chunk_size - 200
        chunks = [text[i:i + chunk_size] for i in range(0, len(text), step)]
        for chunk in chunks:
            chunk_claims = extractor.extract(
                text=chunk,
                source_id=doc.id,
                doc_id=doc.id,
                doc_title=getattr(doc, 'title', None),
            )
            claims.extend(chunk_claims)
    return claims


def process_single_question(record: Dict, output_file: str) -> Dict:
    """
    Exact logic from run_arbgraph.py:process_single_question.
    Difference: retriever built per-record from record['search_results'] (Option A).
    Extra fields from rag_conflicts (conflict_type, correct_answer) passed through.
    """
    query = record['Question']
    qid   = record['id']

    result = {
        'id'       : qid,
        'question' : query,
        # rag_conflicts extra fields (for analysis)
        'conflict_type'  : record.get('conflict_type', ''),
        'correct_answer' : record.get('correct_answer', ''),
        'documents'      : [],
        'arbgraph_meta'  : {
            'rounds'           : 0,
            'validated_claims' : 0,
            'suppressed_claims': 0,
            'retrieval_status' : 'failed',
            'status'           : 'init',
            'error_stage'      : None,
        },
        'answer': ''
    }

    try:
        # Build per-record retriever from pre-loaded passages
        retriever = PreloadedRetriever(
            record.get('search_results', []),
            passage_field=PASSAGE_FIELD
        )
        docs = retriever.retrieve(query, max_docs=MAX_DOCS)

        if not docs:
            result['arbgraph_meta']['status'] = 'no_documents'
        else:
            result['arbgraph_meta']['retrieval_status'] = 'success'
            for idx, doc in enumerate(docs, 1):
                result['documents'].append({
                    'id'     : idx,
                    'title'  : getattr(doc, 'title', ''),
                    'url'    : getattr(doc, 'url', ''),
                    'source' : 'rag_conflicts',
                    'context': getattr(doc, 'text', '')[:3000],
                })

            raw_claims = extract_claims_from_documents(extractor, docs)
            if not raw_claims:
                result['arbgraph_meta']['status'] = 'no_raw_claims'
                result['arbgraph_meta']['error_stage'] = 'claim_extraction'
            else:
                canonical_claims = aligner.align_and_merge(raw_claims)
                if not canonical_claims:
                    result['arbgraph_meta']['status'] = 'no_canonical_claims'
                    result['arbgraph_meta']['error_stage'] = 'claim_alignment'
                else:
                    graph = graph_builder.build(canonical_claims, query=query)
                    arb   = arbitrator.arbitrate(graph, query)

                    validated  = arb.get('validated_claims', [])
                    suppressed = arb.get('suppressed_claims', [])
                    history    = arb.get('arbitration_history', [])

                    result['arbgraph_meta']['validated_claims']  = len(validated)
                    result['arbgraph_meta']['suppressed_claims'] = len(suppressed)
                    result['arbgraph_meta']['rounds']            = len(history)

                    # Lưu validated claims để phân tích sau
                    result['validated_claims'] = [
                        {
                            'text'      : c.get('text', ''),
                            'confidence': c.get('confidence', 0),
                            'evidence'  : c.get('evidence', ''),
                            'doc_title' : c.get('doc_title', ''),
                        }
                        for c in validated
                    ]

                    result['answer'] = generator.generate(
                        query=query,
                        context={'validated_claims': validated},
                    )
                    result['arbgraph_meta']['status'] = 'success'

    except Exception as e:
        result['arbgraph_meta']['status']      = 'exception'
        result['arbgraph_meta']['error_stage'] = 'runtime'
        result['answer'] = f'Error: {str(e)}'

    with open(output_file, 'a', encoding='utf-8') as f:
        f.write(json.dumps(result, ensure_ascii=False) + '\n')

    return result


def load_checkpoint(output_file: str) -> set:
    """Exact copy from run_arbgraph.py"""
    processed = set()
    if os.path.exists(output_file):
        with open(output_file, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    processed.add(json.loads(line)['id'])
                except Exception:
                    continue
    return processed


print('Pipeline functions ready')

## Cell 10 — Run experiment

In [ ]:
from tqdm.auto import tqdm
import time

processed_ids = load_checkpoint(OUTPUT_FILE)
remaining     = [r for r in records if r['id'] not in processed_ids]

print(f'Total    : {len(records)}')
print(f'Done     : {len(processed_ids)}')
print(f'Remaining: {len(remaining)}')
print(f'Output   : {OUTPUT_FILE}\n')

stats = {}

for rec in tqdm(remaining, desc='ArbGraph'):
    t0     = time.time()
    result = process_single_question(rec, OUTPUT_FILE)
    status = result['arbgraph_meta']['status']
    stats[status] = stats.get(status, 0) + 1
    tqdm.write(
        f"  [{result['id']}] "
        f"conflict={result['conflict_type']:<18} "
        f"status={status:<22} "
        f"val={result['arbgraph_meta']['validated_claims']} "
        f"sup={result['arbgraph_meta']['suppressed_claims']} "
        f"({time.time()-t0:.1f}s)"
    )

print('\n' + '='*55)
print('DONE')
for k, v in stats.items():
    print(f'  {k:<25} {v}')

## Cell 11 — Results & analysis by conflict type

In [ ]:
import json, pandas as pd
from collections import defaultdict

results = []
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            results.append(json.loads(line))

rows = []
for r in results:
    m = r.get('arbgraph_meta', {})
    rows.append({
        'id'            : r['id'],
        'conflict_type' : r.get('conflict_type', ''),
        'status'        : m.get('status', ''),
        'validated'     : m.get('validated_claims', 0),
        'suppressed'    : m.get('suppressed_claims', 0),
        'rounds'        : m.get('rounds', 0),
        'answer_len'    : len(r.get('answer', '')),
        'correct_answer': r.get('correct_answer', ''),
        'question'      : r['question'][:55] + '...',
    })

df = pd.DataFrame(rows)

print('=== Per conflict_type summary ===')
print(df.groupby('conflict_type')[['validated','suppressed','rounds','answer_len']]
        .mean().round(2).to_string())

print('\n=== Status distribution ===')
print(df['status'].value_counts().to_string())

print('\n=== Full table (first 20) ===')
print(df.head(20).to_string(index=False))

## Cell 12 — Sample: compare answer vs correct_answer

In [ ]:
success_df = df[df['status'] == 'success'].head(5)
for _, row in success_df.iterrows():
    full = next(r for r in results if r['id'] == row['id'])
    print('='*60)
    print(f"Q            : {full['question']}")
    print(f"conflict_type: {full['conflict_type']}")
    print(f"correct_answer: {full.get('correct_answer','N/A')}")
    print(f"ArbGraph answer:\n{full['answer'][:400]}")
    print()

## Cell 13 — Download output

In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print('Download started (Colab)')
except ImportError:
    print(f'Kaggle: file is at {OUTPUT_FILE}')

## Cell 14 — Trích xuất 10 câu hỏi: validated claims + câu trả lời LLM

In [ ]:
import json

results = []
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            results.append(json.loads(line))

print(f'Records: {len(results)}')
print('=' * 80)

for i, r in enumerate(results, 1):
    m       = r.get('arbgraph_meta', {})
    claims  = r.get('validated_claims', [])
    answer  = r.get('answer', '').strip()

    print(f'[{i:02d}] {r["id"]}')
    print(f'  Question      : {r["question"]}')
    print(f'  conflict_type : {r.get("conflict_type", "N/A")}')
    print(f'  correct_answer: {r.get("correct_answer", "N/A")}')
    print(f'  status        : {m.get("status")}  '
          f'| validated={m.get("validated_claims",0)} '
          f'suppressed={m.get("suppressed_claims",0)} '
          f'rounds={m.get("rounds",0)}')

    # ── Validated claims ──────────────────────────────────────────────
    if claims:
        print(f'  Validated claims ({len(claims)}):')
        for j, c in enumerate(sorted(claims, key=lambda x: -x.get('confidence', 0)), 1):
            conf  = c.get('confidence', 0)
            text  = c.get('text', '').strip()
            src   = c.get('doc_title', '') or c.get('evidence', '')[:60]
            print(f'    [{j}] [{conf:.2f}] {text}')
            if src:
                print(f'         src: {src}')
    else:
        print(f'  Validated claims: (none saved — re-run pipeline to capture)')

    # ── LLM Answer ────────────────────────────────────────────────────
    print(f'  LLM Answer:')
    if answer and not answer.startswith('Error'):
        for line in answer.split('\n'):
            print(f'    {line}')
    else:
        print(f'    (none — status={m.get("status")})')

    print('-' * 80)


## Cell 15 — Evaluation Metrics

| Metric | Cần gì | Cách tính |
|--------|--------|-----------|
| **FR** (Fact Recall) | `correct_answer` → atomic facts (LLM) | facts_in_answer / total_facts |
| **ID** (Info Density) | validated claims count + answer token count | claims / tokens |
| **Faithfulness** | RAGChecker + OpenAI key | claims supported by retrieved docs |
| **Context Utilization** | RAGChecker + OpenAI key | GT facts captured by retrieved docs |
| **Noise-S / Noise-I** | RAGChecker + OpenAI key | sensitivity to noisy evidence |
| **Hallucination** | RAGChecker + OpenAI key | unsupported claims / total claims |
| **Self-knowledge** | RAGChecker + OpenAI key | correct claims from parametric knowledge |

**Cell 15a** — FR + ID (chạy được ngay, dùng LLM local)  
**Cell 15b** — RAGChecker metrics (cần `OPENAI_API_KEY`)

In [ ]:
# ═══════════════════════════════════════════════════════
# Nhóm 1: FR (Fact Recall) + ID (Information Density)
# Không cần OpenAI key — dùng LLM local (llm object đã load)
# ═══════════════════════════════════════════════════════
import json, re
from tqdm.auto import tqdm


def extract_atomic_facts(llm, text: str) -> list[str]:
    """Dùng LLM local để extract atomic facts từ ground-truth answer."""
    if not text or text.strip() == 'N/A':
        return []
    prompt = (
        "Decompose the following answer into a list of atomic factual claims.\n"
        "Each claim must be a single verifiable fact.\n"
        "Return ONLY a JSON array of strings, e.g.: [\"fact 1\", \"fact 2\"]\n\n"
        f"Answer: {text}"
    )
    raw = llm.generate(prompt, max_new_tokens=300, temperature=0.0)
    # parse JSON array
    try:
        m = re.search(r'\[.*?\]', raw, re.S)
        if m:
            return [f.strip() for f in json.loads(m.group(0)) if f.strip()]
    except Exception:
        pass
    # fallback: split by newline/bullet
    lines = [l.strip().lstrip('-•*0123456789.). ') for l in raw.split('\n') if l.strip()]
    return [l for l in lines if len(l) > 10]


def claim_covered(llm, fact: str, answer: str) -> bool:
    """Kiểm tra fact có được cover trong answer không."""
    prompt = (
        f"Does the following answer contain or support this fact?\n"
        f"Fact: {fact}\n"
        f"Answer: {answer[:800]}\n\n"
        "Reply with ONLY 'yes' or 'no'."
    )
    raw = llm.generate(prompt, max_new_tokens=5, temperature=0.0).strip().lower()
    return raw.startswith('yes')


def count_tokens(text: str) -> int:
    """Approximate token count (split by whitespace)."""
    return len(text.split())


# ── Load results ──────────────────────────────────────────────────────────
results = []
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            results.append(json.loads(line))

# Filter chỉ lấy success records có correct_answer
eval_records = [
    r for r in results
    if r.get('arbgraph_meta', {}).get('status') == 'success'
    and r.get('answer', '').strip()
]
print(f'Records for evaluation: {len(eval_records)} / {len(results)}')

# ── Compute FR + ID per record ────────────────────────────────────────────
fr_scores, id_scores = [], []
per_record_metrics = []

for r in tqdm(eval_records, desc='FR + ID'):
    answer       = r.get('answer', '').strip()
    gt_answer    = r.get('correct_answer', '').strip()
    n_validated  = r.get('arbgraph_meta', {}).get('validated_claims', 0)
    n_tokens     = count_tokens(answer)

    # ── ID: validated_claims / answer_tokens ─────────────────────────────
    id_score = n_validated / n_tokens if n_tokens > 0 else 0.0
    id_scores.append(id_score)

    # ── FR: extract GT atomic facts, check coverage ───────────────────────
    if gt_answer and gt_answer != 'N/A':
        gt_facts = extract_atomic_facts(llm, gt_answer)
        if gt_facts:
            covered = sum(claim_covered(llm, f, answer) for f in gt_facts)
            fr = covered / len(gt_facts)
        else:
            fr = 0.0
            gt_facts = []
    else:
        fr = None   # no GT available
        gt_facts = []

    if fr is not None:
        fr_scores.append(fr)

    per_record_metrics.append({
        'id'           : r['id'],
        'conflict_type': r.get('conflict_type', ''),
        'fr'           : round(fr, 3) if fr is not None else None,
        'id_score'     : round(id_score, 4),
        'n_gt_facts'   : len(gt_facts),
        'n_validated'  : n_validated,
        'n_tokens'     : n_tokens,
    })

# ── Summary ───────────────────────────────────────────────────────────────
import pandas as pd
df_m = pd.DataFrame(per_record_metrics)

print('\n=== FR + ID Results ===')
print(df_m.to_string(index=False))

print('\n=== Aggregate (mean) ===')
valid_fr = [x for x in fr_scores if x is not None]
print(f'  Fact Recall (FR)     : {sum(valid_fr)/len(valid_fr):.3f}  (over {len(valid_fr)} records with GT)')
print(f'  Info Density (ID)    : {sum(id_scores)/len(id_scores):.4f}')

if len(df_m) > 0 and 'conflict_type' in df_m.columns:
    print('\n=== Per conflict_type ===')
    print(df_m.groupby('conflict_type')[['fr','id_score']].mean().round(3).to_string())


### Cell 15b — RAGChecker metrics (Faithfulness, Context Utilization, Noise-S/I, Hallucination, Self-knowledge)
Cần `OPENAI_API_KEY`. Bỏ qua nếu không có.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Nhóm 2+3+Bonus: RAGChecker metrics
# Model: Gemini 2.5 Flash (free tier — lấy key tại aistudio.google.com)
# RAGChecker dùng litellm → prefix 'gemini/' + GEMINI_API_KEY
# ═══════════════════════════════════════════════════════════════
import os, json, subprocess, sys

# ── 1. Install ────────────────────────────────────────────────────────────
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                        'ragchecker', 'litellm'])
subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'])

# ── 2. Set Gemini API key ─────────────────────────────────────────────────
# Lấy key miễn phí: https://aistudio.google.com/app/apikey
# Colab  : 🔑 Secrets (góc trái) → Add secret → GEMINI_API_KEY
# Kaggle : Add-ons → Secrets → GEMINI_API_KEY
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('API key loaded from Colab Secrets')
except Exception:
    pass  # Kaggle: key đã có trong env

if not os.environ.get('GEMINI_API_KEY'):
    raise EnvironmentError(
        'GEMINI_API_KEY not set.\n'
        'Lấy key free tại: https://aistudio.google.com/app/apikey\n'
        'Colab  : 🔑 Secrets (góc trái) → Add secret → GEMINI_API_KEY\n'
        'Kaggle : Add-ons → Secrets → GEMINI_API_KEY'
    )
print(f'Key set: ...{os.environ["GEMINI_API_KEY"][-6:]}')

# litellm cần GOOGLE_API_KEY alias để gọi Google AI Studio
os.environ['GOOGLE_API_KEY'] = os.environ['GEMINI_API_KEY']

# ── 3. Build RAGChecker input ─────────────────────────────────────────────
results = []
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            results.append(json.loads(line))

checker_inputs = []
for r in results:
    if r.get('arbgraph_meta', {}).get('status') != 'success':
        continue
    if not r.get('answer', '').strip():
        continue
    retrieved_context = [
        {'doc_id': str(d.get('id', '')), 'text': d.get('context', '')}
        for d in r.get('documents', [])
        if d.get('context', '').strip()
    ]
    checker_inputs.append({
        'query_id'         : str(r['id']),
        'query'            : r['question'],
        'gt_answer'        : r.get('correct_answer', '') or r['question'],
        'response'         : r['answer'],
        'retrieved_context': retrieved_context,
    })

ragchecker_input_file  = '/content/outputs/rag_conflicts/ragchecker_input.json'
ragchecker_output_file = '/content/outputs/rag_conflicts/ragchecker_output.json'

with open(ragchecker_input_file, 'w') as f:
    json.dump({'results': checker_inputs}, f, indent=2, ensure_ascii=False)
print(f'Input : {len(checker_inputs)} records → {ragchecker_input_file}')

# ── 4. Run RAGChecker với Gemini 2.5 Flash ────────────────────────────────
# litellm format: 'gemini/<model-name>'
# gemini-2.5-flash: free tier, nhanh, context window 1M tokens
GEMINI_MODEL = 'gemini/gemini-2.5-flash'

from ragchecker import RAGResults, RAGChecker
from ragchecker.metrics import all_metrics

with open(ragchecker_input_file) as f:
    rag_results = RAGResults.from_dict(json.load(f))

checker = RAGChecker(
    extractor_name=GEMINI_MODEL,
    checker_name=GEMINI_MODEL,
    batch_size_extractor=4,
    batch_size_checker=4,
)
checker.evaluate(rag_results, all_metrics)

# ── 5. Save + print ───────────────────────────────────────────────────────
out = rag_results.to_dict()
with open(ragchecker_output_file, 'w') as f:
    json.dump(out, f, indent=2, ensure_ascii=False)

overall = out.get('overall_metrics', {})

metric_map = {
    'faithfulness'                   : 'Faithfulness',
    'context_utilization'            : 'Context Utilization',
    'noise_sensitivity_in_relevant'  : 'Noise-S (Relevant)',
    'noise_sensitivity_in_irrelevant': 'Noise-I (Irrelevant)',
    'hallucination'                  : 'Hallucination',
    'self_knowledge'                 : 'Self-knowledge',
}

print(f'\n=== RAGChecker Metrics (model: {GEMINI_MODEL}) ===')
print(f'{"Metric":<30} {"Score"}')
print('-' * 42)
for key, label in metric_map.items():
    val = overall.get(key, 'N/A')
    val_str = f'{val:.3f}' if isinstance(val, (int, float)) else str(val)
    print(f'  {label:<28} {val_str}')

print(f'\nFull output → {ragchecker_output_file}')

# ── 6. Per conflict_type breakdown ────────────────────────────────────────
import pandas as pd

id2conflict = {str(r['id']): r.get('conflict_type', '') for r in results}
rows = []
for res in out.get('results', []):
    qid = str(res.get('query_id', ''))
    m   = res.get('metrics', {})
    rows.append({
        'conflict_type' : id2conflict.get(qid, ''),
        'faithfulness'  : m.get('faithfulness'),
        'context_util'  : m.get('context_utilization'),
        'noise_s'       : m.get('noise_sensitivity_in_relevant'),
        'noise_i'       : m.get('noise_sensitivity_in_irrelevant'),
        'hallucination' : m.get('hallucination'),
        'self_knowledge': m.get('self_knowledge'),
    })

df_rc = pd.DataFrame(rows)
print('\n=== Per conflict_type (RAGChecker) ===')
print(df_rc.groupby('conflict_type').mean().round(3).to_string())


### Cell 15c — Combined summary: tất cả 8 metrics

In [ ]:
# Chạy sau cả 15a và 15b
# Tổng hợp tất cả metrics vào 1 bảng
import pandas as pd

summary = {
    'Metric'   : ['FR (Fact Recall)', 'ID (Info Density)',
                  'Faithfulness', 'Context Utilization',
                  'Noise-S', 'Noise-I',
                  'Hallucination', 'Self-knowledge'],
    'Group'    : ['Generative', 'Generative',
                  'Factual Grounding', 'Factual Grounding',
                  'Robustness', 'Robustness',
                  'Attribution', 'Attribution'],
    'Source'   : ['LLM local', 'Count-based',
                  'RAGChecker', 'RAGChecker',
                  'RAGChecker', 'RAGChecker',
                  'RAGChecker', 'RAGChecker'],
    'Direction': ['↑ higher better', '↑ higher better',
                  '↑ higher better', '↑ higher better',
                  '↓ lower better', '↓ lower better',
                  '↓ lower better', '↑ higher better'],
    'Score'    : [
        round(sum(fr_scores)/len(fr_scores), 3) if fr_scores else 'N/A',
        round(sum(id_scores)/len(id_scores), 4) if id_scores else 'N/A',
        overall.get('faithfulness', 'N/A'),
        overall.get('context_utilization', 'N/A'),
        overall.get('noise_sensitivity_in_relevant', 'N/A'),
        overall.get('noise_sensitivity_in_irrelevant', 'N/A'),
        overall.get('hallucination', 'N/A'),
        overall.get('self_knowledge', 'N/A'),
    ]
}

df_all = pd.DataFrame(summary)
print('\n=== ArbGraph × RAG-Conflicts — Full Evaluation ===')
print(f'  Dataset : rag_conflicts  ({len(eval_records)} success records)')
print(f'  Model   : {MODEL_NAME}')
print()
print(df_all.to_string(index=False))
